In [1]:
%matplotlib widget
#%matplotlib notebook
import numpy as np
from ucimlrepo import fetch_ucirepo
from praxis import PRAXIS, ThresholdGuessBinarizer


def accuracy(y_true, y_pred):
    return (np.asarray(y_true).astype(int) == np.asarray(y_pred).astype(int)).mean()


magic = fetch_ucirepo(id=159)
X_raw = magic.data.features.copy()
y_raw = magic.data.targets.copy()

y_col = y_raw.columns[0]
y = (y_raw[y_col] == "g").to_numpy(np.int64)


enc = ThresholdGuessBinarizer(
    n_estimators=75,
    max_depth=2,
    random_state=0,
    column_elimination=False,
)

X = enc.fit_transform(X_raw, y).astype(np.uint8)

binning_map = enc.feature_map()

print("Original X shape: ", X_raw.shape)
print("Binarized X shape:", X.shape)
print("Binning map:", binning_map)


model = PRAXIS()

model.fit(
    X, y,
    lambda_reg=0.005,
    depth_budget=5,
    rashomon_mult=0.01,
    multiplicative_slack=0,
    key_mode="hash",
    lookahead_k=1,
)

print("\nMinimum objective:", model.get_min_objective())
print("Rashomon set size:", model.count_trees())


binary_feature_names = enc.get_feature_names_out()
raw_feature_names = list(enc.feature_names_in_)

continuous_groups = {
    raw_feature_names[int(raw_j)]: [int(c) for c in bin_cols]
    for raw_j, bin_cols in binning_map.items()
}

builder = model.interactive_tree_builder(
    feature_names=binary_feature_names,
    continuous_groups=continuous_groups,
    figsize=(6, 3),
    auto_expand_single=True,
    title="Interactive PRAXIS Tree Builder",
)




Original X shape:  (19020, 10)
Binarized X shape: (19020, 160)
Binning map: {0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27], 1: [28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53], 2: [54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94], 3: [95, 96], 4: [97, 98, 99, 100, 101, 102, 103, 104, 105], 5: [106, 107], 6: [108, 109, 110, 111, 112, 113, 114, 115, 116], 7: [117, 118], 8: [119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144], 9: [145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159]}
Best objective: 3920
Minimum objective: 3905
Rashomon set size: 782076
 (0.206099)
Objective bound: 3959
Minimum objective: 3905
Cache sizes - Greedy: 774377, Lickety: 6909516, Trie: 